# Evaluate Metrics
This notebook evaluates metrics using IBM watsonx.governance SDK. RAG metrics evaluation is shown for demonstration. RAG metrics are evaluated by taking in the data containing contexts, question, answer and ground truth (Optional) information. The metrics result will be visualized using the `ModelInsights`.

This notebook should be run in Python 3.10 or greater runtime environment.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### Configure your credentials using Credentials object

#### Using watsonx.governance or watsonx.ai as Service
These are the needed values when using watsonx.governance as service:
- `region`: This is the region for the watsonx.governance as service. This field is optional; by default, it is set to the us-south(Dallas) region. Supported region values are us-south, eu-de, au-syd, ca-tor, jp-tok, eu-gb.
- `api_key`: The API key required for authentication. Instructions for generating API keys can be found
[here](https://cloud.ibm.com/docs/account?topic=account-userapikey&interface=ui).
- `service_instance_id`: This is the instance ID for watsonx.governance as service. To retrieve these details, follow the steps in next section.

##### Retrieving `service_instance_id` value
You can view the service instance ID that is associated with your watsonx.governance service instance by navigating to your IBM Cloud resource list.
1. [Log in to IBM Cloud](https://cloud.ibm.com/).
2. Go to Menu > Resource list, and then click Services to browse a list of your cloud services.
3. Click the table row that describes your watsonx.governance service instance.
4. Go to Overview > Instance, and copy the GUID value.

In [ ]:
from ibm_watsonx_gov.config import Credentials
from ibm_watsonx_gov.entities.enums import Region
import os


credentials = Credentials(
    api_key=os.getenv("WATSONX_APIKEY"),
    service_instance_id=os.getenv("SERVICE_INSTANCE_ID"),
    # region=Region.AU_SYD.value #optional. Uncomment to specify a different region.
)

## Evaluate Metrics<a name="evaluate"></a>

In this step the input to the RAG application, its generated output, and the retrieved contexts are loaded. To get started, please use the sample CSV dataset file provided in this notebook. However, if you want to use your own RAG application and would like to evaluate it's output, the dataset can be loaded in the following cell and the configuration can be updated to read the specific columns in the dataset.

### Loading your test data

In this step, you can load your own sample RAG dataset in `input_df` and update the configuration object in the next step to match the column names from the dataset.

In [3]:
import pandas as pd
input_df = pd.read_csv("https://raw.githubusercontent.com/IBM/ibm-watsonx-gov/refs/heads/samples/notebooks/data/rag/rag_with_ground_truth.csv")
input_df.head()

,question,contexts,answer,ground_truth
0,What is the capital of France?,"['Paris is the capital of France.', 'France is...",Paris,Paris
1,Who wrote '1984'?,"['George Orwell wrote the book 1984.', '1984 i...",George Orwell,George Orwell
2,What is the largest mammal?,['The blue whale is the largest mammal on Eart...,Elephant,Blue Whale
3,Where is the Great Wall located?,"['The Great Wall of China is in Asia.', 'It sp...",China,India
4,What is the boiling point of water?,['Water boils at 100 degrees Celsius at sea le...,100 degrees Celsius,100 degrees Celsius


### Configure the evaluator

Once the dataset is loaded, need to configure the evaluator to specify what are the columns of interests in the data set and to specify which metrics to evaluate

#### Dataset columns

To configure the evaluator for the dataset, please update `question_field` to be the column name that contains the questions, `context_field` to be a list of column names that contains the contexts, and `output_fields` to have the column that contain the generated answer. Optionally, you can add the ground truth column name under `reference_fields`.

#### Metrics

You can provide the metrics to evaluate in the `metrics` list. The metric evaluation method(i.e. the technique used to compute the metric) can be provided by specifying the `method` value when creating the metric object. In addition, you can also provide the metric groups under `metric_groups` which will evaluate all the metrics belonging to the specified group(s).

The complete list of available metrics can be referred [here](https://ibm.github.io/ibm-watsonx-gov/generated_apidoc/ibm_watsonx_gov.metrics.html)

The complete list of available metric groups can be referred [here](https://ibm.github.io/ibm-watsonx-gov/generated_apidoc/ibm_watsonx_gov.entities.enums.html#ibm_watsonx_gov.entities.enums.MetricGroup)

In [4]:
from ibm_watsonx_gov.config import GenAIConfiguration
from ibm_watsonx_gov.metrics import ContextRelevanceMetric, FaithfulnessMetric, AnswerSimilarityMetric, AnswerRelevanceMetric
from ibm_watsonx_gov.entities.enums import MetricGroup

question_field = "question"
context_field = "contexts"

config = GenAIConfiguration(
    input_fields=[question_field],
    context_fields=[context_field],
    output_fields=["answer"],
    reference_fields=["ground_truth"]
)

metrics = [
    ContextRelevanceMetric(),
    FaithfulnessMetric(),
    AnswerSimilarityMetric(),
    AnswerRelevanceMetric(),
]

metric_groups = [
    MetricGroup.RETRIEVAL_QUALITY,
    MetricGroup.ANSWER_QUALITY,
    MetricGroup.CONTENT_SAFETY
]

Optional dependency for IBM Watson not found: No module named 'ibm_watsonx_ai'


### Run the metrics evaluation

Create `MetricEvaluator` instance using the configuration and credentials. After that compute the desired metrics by invoking `evaluate()` method.

In [5]:
from ibm_watsonx_gov.clients.api_client import APIClient
from ibm_watsonx_gov.evaluators import MetricsEvaluator

evaluator = MetricsEvaluator(
    #api_client=APIClient(credentials=credentials), # Uncomment this line when using Credentials object
    configuration=config,
)

evaluation_result = evaluator.evaluate(
    data=input_df,
    metrics=metrics,
    metric_groups=metric_groups
)

[Warning] No region provided : Using default region as us-south


## Display the results  <a name="display"></a>

### Display the result table

Now that the evaluation is done, display a table containing each record with its metric values.

In [6]:
evaluation_result.to_df()

,context_relevance.token_precision,faithfulness.token_k_precision,answer_similarity.token_recall,answer_relevance.token_recall,unsuccessful_requests,evasiveness.granite_guardian,hap,input_hap,output_hap,harm.granite_guardian,...,profanity.granite_guardian,sexual_content.granite_guardian,social_bias.granite_guardian,unethical_behavior.granite_guardian,violence.granite_guardian,average_precision,hit_rate,ndcg,reciprocal_rank,retrieval_precision
0,0.8000,1.0,1.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
1,0.6667,1.0,1.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.7500,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
3,0.6000,1.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.6667,1.0,1.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
5,0.8000,1.0,1.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
6,0.5000,1.0,1.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
7,0.7500,0.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
8,0.4000,1.0,1.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
9,0.8333,1.0,0.0,0.0,0.0,None,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,1.0
